# Athena emulator — analytics dataframes

You're an analyst or BI engineer who needs query results as pandas
DataFrames. In a handful of cells you'll read the same data three ways —
csv artifact, CTAS parquet, managed-results inline — then shape reads with
cache, chunking, typed columns, and parameters, and finish with the
table/execution/schema helpers.

Stack (`docker-compose.yml`, network `mlops_net`): `athena` (emulator) ·
`trino` (query engine) · `moto` (S3/Glue) · `jupyterlab` (this notebook).
Per ADR-0002, only Athena traffic goes to the emulator; S3/Glue calls must
still land on moto.


In [1]:
import os

# Compose sets these for in-container runs; the localhost fallbacks let the
# same notebook run against the published ports from the host.
os.environ.setdefault("AWS_ENDPOINT_URL", "http://localhost:5000")
os.environ.setdefault("AWS_ENDPOINT_URL_ATHENA", "http://localhost:5001")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ATHENA_URL = os.environ["AWS_ENDPOINT_URL_ATHENA"]
EDGE_URL = os.environ["AWS_ENDPOINT_URL"]

print(f"athena → {ATHENA_URL}")
print(f"s3/glue → {EDGE_URL}")

athena → http://localhost:5001
s3/glue → http://localhost:5000


In [2]:
import time
import uuid

import awswrangler as wr
import boto3
import pandas as pd
from _evidence import persist, record, render
from _helpers import (
    delete_run_buckets,
    delete_run_databases,
    run_workgroup_names,
)

athena = boto3.client("athena")
s3 = boto3.client("s3")
glue = boto3.client("glue")

assert athena.meta.endpoint_url.rstrip("/") == ATHENA_URL, (
    athena.meta.endpoint_url
)
assert s3.meta.endpoint_url.rstrip("/") == EDGE_URL, s3.meta.endpoint_url
assert glue.meta.endpoint_url.rstrip("/") == EDGE_URL, glue.meta.endpoint_url

RUN_SUFFIX = uuid.uuid4().hex[:8]
# Every resource this run creates carries the nb02- prefix (nb02_ for Glue
# databases and tables, where '-' is invalid) so cleanup can discover
# leftovers — its own and any failed run's — instead of tracking a list.
ATHENA_RESOURCE_PREFIX = "nb02-"
DATABASE_PREFIX = "nb02_"
# Buckets that existed before this run: the wrangler default results bucket
# is only cleaned up when this run's create_athena_bucket call created it.
pre_existing_buckets = {info["Name"] for info in s3.list_buckets()["Buckets"]}

bucket = f"{ATHENA_RESOURCE_PREFIX}{uuid.uuid4().hex[:12]}"
database = f"{DATABASE_PREFIX}{RUN_SUFFIX}"
managed_workgroup = f"{ATHENA_RESOURCE_PREFIX}mgr-{RUN_SUFFIX}"
people_table = f"{DATABASE_PREFIX}people_{RUN_SUFFIX}"
sales_table = f"{DATABASE_PREFIX}sales_{RUN_SUFFIX}"
results_prefix = f"s3://{bucket}/results/"

print(f"bucket={bucket} database={database} workgroup={managed_workgroup}")

bucket=nb02-6ec984600928 database=nb02_b8c9af57 workgroup=nb02-mgr-b8c9af57


## The data you'll read

Setup, not a demonstrated capability: two small parquet datasets registered
in the Glue catalog, plus a managed-results workgroup for the inline-read
demo below.


In [3]:
s3.create_bucket(Bucket=bucket)
glue.create_database(DatabaseInput={"Name": database})

people = pd.DataFrame(
    {
        "id": [1, 2, 3],
        "name": ["alpha", "beta", "gamma"],
        "score": [9.5, 8.25, 7.0],
    }
)
wr.s3.to_parquet(
    df=people,
    path=f"s3://{bucket}/datasets/people/",
    dataset=True,
    database=database,
    table=people_table,
)

sales = pd.DataFrame(
    {
        "quantity": [3, 7, 5],
        "region": ["US", "EU", "US"],
        "amount": [10.5, 3.14, 3.14],
    }
)
wr.s3.to_parquet(
    df=sales,
    path=f"s3://{bucket}/datasets/sales/",
    dataset=True,
    database=database,
    table=sales_table,
    partition_cols=["region"],
)

# Managed results: GetQueryExecution reports an empty ResultConfiguration and
# wrangler falls back to inline GetQueryResults reads (the api path).
athena.create_work_group(
    Name=managed_workgroup,
    Configuration={"ManagedQueryResultsConfiguration": {"Enabled": True}},
)
print(f"registered {people_table}, {sales_table} in {database}")

registered nb02_people_b8c9af57, nb02_sales_b8c9af57 in nb02_b8c9af57


## Read it three ways

`read_sql_query` returns your query as a DataFrame — the csv approach reads
the `{qid}.csv` artifact, ctas (default) reads manifest-listed parquet, and
the api approach (managed-results workgroup) serves rows inline.


In [4]:
started = time.monotonic()
csv_frame = wr.athena.read_sql_query(
    sql=f'SELECT * FROM "{database}"."{people_table}" ORDER BY id',
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
)
csv_qid = csv_frame.query_metadata["QueryExecutionId"]
assert csv_frame["name"].tolist() == ["alpha", "beta", "gamma"]
record(
    "read_sql_query csv path reads a wrangler-registered table",
    "PASS",
    f"{csv_qid} shape={csv_frame.shape} — Trino resolved the "
    "Glue-registered parquet table",
    (time.monotonic() - started) * 1000,
)
csv_frame

,id,name,score
0,1,alpha,9.50
1,2,beta,8.25
2,3,gamma,7.00


In [5]:
started = time.monotonic()
ctas_frame = wr.athena.read_sql_query(
    sql="SELECT 1 AS one, 'alpha' AS label",
    database=database,
    s3_output=results_prefix,
)
ctas_qid = ctas_frame.query_metadata["QueryExecutionId"]
ctas_execution = athena.get_query_execution(QueryExecutionId=ctas_qid)[
    "QueryExecution"
]
assert ctas_frame.loc[0, "one"] == 1
assert ctas_execution["StatementType"] == "DDL"
assert ctas_execution["SubstatementType"] == "CREATE_TABLE_AS_SELECT"
ctas_manifest = ctas_execution["Statistics"]["DataManifestLocation"]
assert ctas_manifest.endswith("-manifest.csv")
record(
    "read_sql_query ctas_approach (temp CTAS → manifest → parquet read)",
    "PASS",
    f"{ctas_qid} rows={len(ctas_frame)}; "
    f"manifest={ctas_manifest.split('/')[-1]}",
    (time.monotonic() - started) * 1000,
)

Evidence(feature='read_sql_query ctas_approach (temp CTAS → manifest → parquet read)', status='PASS', detail='c91dd96d-57b3-419a-9c7b-1a758e7b8959 rows=1; manifest=c91dd96d-57b3-419a-9c7b-1a758e7b8959-manifest.csv', latency_ms=1101.5721319708973)

In [6]:
started = time.monotonic()
api_frame = wr.athena.read_sql_query(
    sql="SELECT 2 AS two, 'beta' AS label",
    database=database,
    ctas_approach=False,
    workgroup=managed_workgroup,
)
api_qid = api_frame.query_metadata["QueryExecutionId"]
managed_execution = athena.get_query_execution(QueryExecutionId=api_qid)[
    "QueryExecution"
]
assert api_frame.loc[0, "two"] == 2
# Managed results report an empty ResultConfiguration — the signal wrangler
# uses to fall back to inline GetQueryResults.
assert managed_execution["ResultConfiguration"] == {}
record(
    "read_sql_query api path (managed workgroup → inline GetQueryResults)",
    "PASS",
    f"{api_qid} ResultConfiguration={{}}; {len(api_frame)} row(s) served "
    "inline",
    (time.monotonic() - started) * 1000,
)

Evidence(feature='read_sql_query api path (managed workgroup → inline GetQueryResults)', status='PASS', detail='a2ecd491-0eb4-498f-9508-9d173b4ce33e ResultConfiguration={}; 1 row(s) served inline', latency_ms=1083.2202520687133)

## Cache and fetch options

`athena_cache_settings` reuses a recent identical execution; `categories`,
`chunksize`, and `dtype_backend` shape the returned frames; `params` formats
client-side (`named`) or binds server-side (`qmark`).


In [7]:
cache_sql = "SELECT 7 AS seven, 'cached' AS label"
cache_settings = {"max_cache_seconds": 90}
cache_first = wr.athena.read_sql_query(
    sql=cache_sql,
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
    athena_cache_settings=cache_settings,
)
cache_first_id = cache_first.query_metadata["QueryExecutionId"]
cache_second = wr.athena.read_sql_query(
    sql=cache_sql,
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
    athena_cache_settings=cache_settings,
)
cache_second_id = cache_second.query_metadata["QueryExecutionId"]
# A hit re-reads the first execution's csv instead of running again.
assert cache_second_id == cache_first_id
record(
    "athena_cache_settings cache hit reuses the first execution",
    "PASS",
    f"both reads → {cache_first_id}; list+batch_get supplied the hit",
)

Evidence(feature='athena_cache_settings cache hit reuses the first execution', status='PASS', detail='both reads → a0acb8f0-7225-426d-8cc7-711009e628b9; list+batch_get supplied the hit', latency_ms=None)

In [8]:
cat_frame = wr.athena.read_sql_query(
    sql=f'SELECT region, quantity FROM "{database}"."{sales_table}"',
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
    categories=["region"],
)
assert str(cat_frame["region"].dtype) == "category"
record(
    "read_sql_query categories → pandas Categorical",
    "PASS",
    f"region dtype={cat_frame['region'].dtype}",
)

chunks = list(
    wr.athena.read_sql_query(
        sql=f'SELECT * FROM "{database}"."{people_table}" ORDER BY id',
        database=database,
        ctas_approach=False,
        s3_output=results_prefix,
        chunksize=2,
    )
)
chunk_sizes = [len(chunk) for chunk in chunks]
assert chunk_sizes == [2, 1]
record(
    "read_sql_query chunksize=2 → iterator of row-bounded frames",
    "PASS",
    f"chunk row counts={chunk_sizes}",
)

pyarrow_frame = wr.athena.read_sql_query(
    sql="SELECT 1 AS one, 1.5 AS two",
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
    dtype_backend="pyarrow",
)
pyarrow_dtypes = [str(dtype) for dtype in pyarrow_frame.dtypes]
assert all(dtype.endswith("[pyarrow]") for dtype in pyarrow_dtypes)
record(
    'read_sql_query dtype_backend="pyarrow"',
    "PASS",
    f"dtypes={pyarrow_dtypes}",
)

Evidence(feature='read_sql_query dtype_backend="pyarrow"', status='PASS', detail="dtypes=['int32[pyarrow]', 'double[pyarrow]']", latency_ms=None)

In [9]:
named_frame = wr.athena.read_sql_query(
    sql=(
        f'SELECT quantity FROM "{database}"."{sales_table}" '
        "WHERE region = :region"
    ),
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
    params={"region": "EU"},
)
assert named_frame["quantity"].tolist() == [7]
record(
    "read_sql_query params (named → client-side format)",
    "PASS",
    "WHERE region = 'EU' pruned read → 1 row",
)

qmark_frame = wr.athena.read_sql_query(
    sql=(
        f'SELECT quantity FROM "{database}"."{sales_table}" WHERE region = ?'
    ),
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
    params=["EU"],
    paramstyle="qmark",
)
assert qmark_frame["quantity"].tolist() == [7]
qmark_qid = qmark_frame.query_metadata["QueryExecutionId"]
qmark_echoed = athena.get_query_execution(QueryExecutionId=qmark_qid)[
    "QueryExecution"
].get("ExecutionParameters")
assert qmark_echoed == ["EU"]
record(
    "read_sql_query params (qmark → server-side ExecutionParameters)",
    "PASS",
    f"{qmark_qid} bound server-side → 1 row; "
    f"ExecutionParameters echoed={qmark_echoed}",
)

Evidence(feature='read_sql_query params (qmark → server-side ExecutionParameters)', status='PASS', detail="307ba069-6f6d-4226-b4b5-f8e42081329a bound server-side → 1 row; ExecutionParameters echoed=['EU']", latency_ms=None)

## Table and result helpers

`read_sql_table` wraps a `SELECT *` in `read_sql_query`; `get_query_results`
re-reads a finished execution — the csv artifact for a DML `SELECT`, the
manifest-listed parquet for a CTAS execution.


In [10]:
table_frame = wr.athena.read_sql_table(
    table=people_table,
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
)
assert sorted(table_frame["name"].tolist()) == ["alpha", "beta", "gamma"]
record(
    "read_sql_table reads the full registered table",
    "PASS",
    f"shape={table_frame.shape}",
)

Evidence(feature='read_sql_table reads the full registered table', status='PASS', detail='shape=(3, 3)', latency_ms=None)

In [11]:
csv_results = wr.athena.get_query_results(query_execution_id=cache_first_id)
assert csv_results.loc[0, "seven"] == 7
record(
    "get_query_results fetches the csv artifact of a DML SELECT",
    "PASS",
    f"{cache_first_id[:8]}… → shape={csv_results.shape}",
)

ctas_results = wr.athena.get_query_results(query_execution_id=ctas_qid)
assert ctas_results.loc[0, "one"] == 1
record(
    "get_query_results fetches manifest parquet of a CTAS execution",
    "PASS",
    f"{ctas_qid[:8]}… → shape={ctas_results.shape}",
)

Evidence(feature='get_query_results fetches manifest parquet of a CTAS execution', status='PASS', detail='c91dd96d… → shape=(1, 2)', latency_ms=None)

## Execution metadata helpers

Wrangler's metadata calls map onto `ListQueryExecutions`,
`BatchGetQueryExecution`, and `GetQueryExecution` — `wait_query` polls the
last until a terminal state.


In [12]:
listed = wr.athena.list_query_executions(workgroup="primary")
assert cache_first_id in listed
record(
    "list_query_executions returns the workgroup's execution ids",
    "PASS",
    f"{len(listed)} id(s) incl. {cache_first_id[:8]}…",
)

executions_frame, unprocessed = wr.athena.get_query_executions(
    [cache_first_id, ctas_qid, "00000000-0000-0000-0000-000000000000"],
    return_unprocessed=True,
)
assert set(executions_frame["QueryExecutionId"]) == {cache_first_id, ctas_qid}
assert len(unprocessed) == 1
record(
    "get_query_executions batch_gets into frames (+ unprocessed ids)",
    "PASS",
    f"{len(executions_frame)} execution rows + {len(unprocessed)} unprocessed",
)

fetched = wr.athena.get_query_execution(cache_first_id)
assert fetched["Status"]["State"] == "SUCCEEDED"
record(
    "get_query_execution returns the execution member",
    "PASS",
    f"{cache_first_id[:8]}… Status.State={fetched['Status']['State']}",
)

waited = wr.athena.wait_query(cache_first_id)
assert waited["Status"]["State"] == "SUCCEEDED"
record(
    "wait_query returns on a terminal execution",
    "PASS",
    f"{cache_first_id[:8]}… returned SUCCEEDED",
)

Evidence(feature='wait_query returns on a terminal execution', status='PASS', detail='a0acb8f0… returned SUCCEEDED', latency_ms=None)

## Schema helpers

`DESCRIBE` and `SHOW CREATE TABLE` are spelled with backtick-quoted table
names and a statement terminator; `get_query_columns_types` maps the
execution's `ColumnInfo` to `{name: type}`.


In [13]:
described = wr.athena.describe_table(
    table=people_table, database=database, s3_output=results_prefix
)
described_columns = described.iloc[:, 0].tolist()
assert described_columns == ["id", "name", "score"]
record(
    "describe_table returns the table's column listing",
    "PASS",
    f"columns={described_columns}",
)

shown = wr.athena.show_create_table(
    table=people_table, database=database, s3_output=results_prefix
)
assert shown.startswith("CREATE TABLE")
record(
    "show_create_table returns the CREATE TABLE statement",
    "PASS",
    shown.splitlines()[0][:80],
)

Evidence(feature='show_create_table returns the CREATE TABLE statement', status='PASS', detail='CREATE TABLE hive.nb02_b8c9af57.nb02_people_b8c9af57 (', latency_ms=None)

In [14]:
column_types = wr.athena.get_query_columns_types(cache_first_id)
assert column_types == {"seven": "integer", "label": "varchar(6)"}
record(
    "get_query_columns_types maps ColumnInfo Name/Type",
    "PASS",
    f"{column_types}",
)

decimal_frame = wr.athena.read_sql_query(
    sql="SELECT CAST(12.34 AS DECIMAL(6, 2)) AS amount",
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
)
decimal_types = wr.athena.get_query_columns_types(
    decimal_frame.query_metadata["QueryExecutionId"]
)
assert decimal_types == {"amount": "decimal(6, 2)"}
record(
    "get_query_columns_types formats decimal precision/scale",
    "PASS",
    f"{decimal_types}",
)

Evidence(feature='get_query_columns_types formats decimal precision/scale', status='PASS', detail="{'amount': 'decimal(6, 2)'}", latency_ms=None)

## Bucket and workgroup helpers

`create_athena_bucket` derives `aws-athena-query-results-{account}-{region}`
through STS `GetCallerIdentity` (moto answers `123456789012`) and creates it
on moto S3; `get_work_group` reads `primary`'s configuration.


In [15]:
default_path = wr.athena.create_athena_bucket()
default_bucket = default_path.removeprefix("s3://").rstrip("/")
s3.head_bucket(Bucket=default_bucket)
record(
    "create_athena_bucket creates the default results bucket",
    "PASS",
    f"{default_path} exists on moto",
)

primary_wg = wr.athena.get_work_group("primary")
primary_member = primary_wg["WorkGroup"]
assert primary_member["State"] == "ENABLED"
record(
    "get_work_group returns primary's configuration",
    "PASS",
    f"state={primary_member['State']} "
    f"enforce={primary_member['Configuration']['EnforceWorkGroupConfiguration']}",
)

Evidence(feature="get_work_group returns primary's configuration", status='PASS', detail='state=ENABLED enforce=False', latency_ms=None)

## Capabilities demonstrated

Every feature above recorded a measured row. The matrix persists to
`parity/02_analytics_dataframes.md`; `PARITY.md` at the project root is
regenerated from all fragments.


In [16]:
print(render())
persist("02_analytics_dataframes")

| feature | status | detail | latency_ms |
|---|---|---|---|
| read_sql_query csv path reads a wrangler-registered table | PASS | 02ffaeab-1e10-48e7-9c8a-2a7061aa6567 shape=(3, 3) — Trino resolved the Glue-registered parquet table | 1085.71 |
| read_sql_query ctas_approach (temp CTAS → manifest → parquet read) | PASS | c91dd96d-57b3-419a-9c7b-1a758e7b8959 rows=1; manifest=c91dd96d-57b3-419a-9c7b-1a758e7b8959-manifest.csv | 1101.57 |
| read_sql_query api path (managed workgroup → inline GetQueryResults) | PASS | a2ecd491-0eb4-498f-9508-9d173b4ce33e ResultConfiguration={}; 1 row(s) served inline | 1083.22 |
| athena_cache_settings cache hit reuses the first execution | PASS | both reads → a0acb8f0-7225-426d-8cc7-711009e628b9; list+batch_get supplied the hit |  |
| read_sql_query categories → pandas Categorical | PASS | region dtype=category |  |
| read_sql_query chunksize=2 → iterator of row-bounded frames | PASS | chunk row counts=[2, 1] |  |
| read_sql_query dtype_backend="pyarrow" | P

PosixPath('/home/gabriel-menezes/Documents/repos/mlops/projects/athena_emulator/parity/02_analytics_dataframes.md')

## Clean up

The run's bucket, Glue databases, workgroup, and (only if this run created
it) the default results bucket are throwaway — the stack is left as found.


In [17]:
for name in run_workgroup_names(athena, ATHENA_RESOURCE_PREFIX):
    athena.delete_work_group(WorkGroup=name)
# create_athena_bucket() names its bucket after the moto account, so it
# never carries the run prefix — remove it only when this run created
# it (it was absent from the setup-time listing).
delete_run_buckets(
    s3,
    ATHENA_RESOURCE_PREFIX,
    extra=lambda name: (
        name.startswith("aws-athena-query-results-")
        and name not in pre_existing_buckets
    ),
)
delete_run_databases(glue, DATABASE_PREFIX)
print(
    "cleaned: "
    f"workgroups={run_workgroup_names(athena, ATHENA_RESOURCE_PREFIX)} "
    "nb02 buckets+glue databases (incl. tables) and this run's default "
    "results bucket removed"
)

cleaned: workgroups=[] nb02 buckets+glue databases (incl. tables) and this run's default results bucket removed
